In [ ]:
%load_ext autoreload
%autoreload 2
#import os
#os.chdir('/data/luca/lipidatlas/euclid/euclid_msi/tests')

In [ ]:
import os
import pandas as pd
import numpy as np
import scanpy as sc

from euclid_msi.preprocessing import Preprocessing
from euclid_msi.embedding import Embedding
from euclid_msi.clustering import Clustering
from euclid_msi.postprocessing import Postprocessing
from euclid_msi.euclid_casecontrol import CaseControlAnalysis

## Download the dataset from Zenodo

In [ ]:
import os
import tarfile
import requests
from tqdm import tqdm

ZENODO_IDs = ["15650014", "16524009", "16521812"]

for zenodo_id in ZENODO_IDs:
    print(f"--- Processing Zenodo Record: {zenodo_id} ---")

    # 1. Query the Zenodo API to get the list of files for the record
    api_url = f"https://zenodo.org/api/records/{zenodo_id}"
    try:
        response = requests.get(api_url)
        response.raise_for_status()  
        record_data = response.json()
        files_info = record_data.get('files', [])
    except requests.exceptions.RequestException as e:
        print(f"Error querying Zenodo API for record {zenodo_id}: {e}")
        continue  

    if not files_info:
        print(f"No files found for record {zenodo_id}.")
        continue

    # 2. Loop through each file found in the record
    for file_info in files_info:
        filename = file_info['key']
        download_url = file_info['links']['self']

        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            response = requests.get(download_url, stream=True)
            total_size = int(response.headers.get('content-length', 0))

            with open(filename, 'wb') as file:
                with tqdm(total=total_size, unit='B', unit_scale=True, desc=filename) as pbar:
                    for chunk in response.iter_content(chunk_size=8192):
                        file.write(chunk)
                        pbar.update(len(chunk))
            print(f"Download of {filename} completed!")
        else:
            print(f"{filename} already exists.")

        if filename.endswith('.tar.gz'):
            print(f"Extracting {filename}...")
            with tarfile.open(filename, 'r:gz') as tar:
                tar.extractall(path='.')
                print(f"  Extraction completed!")
        else:
            print(f"Skipping extraction for {filename} (not a .tar.gz file).")

    print(f"\n--- Finished processing record {zenodo_id} ---\n")

print("All datasets ready for use! ✅")

## Dashboard

In [ ]:
# the acquisitions metadata file, which has columns SectionID, Path, Sample, Sex, Condition, Section, BadSection
# SectionID is an arbitary numeric section identifier (eg sequence of acquisitions in time)
# Path refers to the relative path to the uMAIA-normalized (or also raw if you like risk) ZARR file
# Sample is the sample identifier (eg, different animals)
# Sex is the sex of the sample
# Condition is the condition of the sample (eg, naive, treated, diseased...)
# Section is the within-individual section number along the rostrocaudal axis, should be consistent
# BadSection is a boolean indicating if the section is corrupted
metadata_csv_file = "acquisitions_metadata.csv"

# the acquisition paths file, which has columns acqn, acqpath
# it is just a sorted list of the acquisitions available in the uMAIA-normalized ZARR file with their relative path
acquisition_paths_csv = "acquisitionpaths.csv"

# the path to the uMAIA-normalized ZARR files (don't use the trailing .zarr, just the name)
uMAIA_path = './uMAIA_subset'

# a parquet file containing additional pixel-level metadata from independent pipelines, such as anatomical annotations
# it must have columns x, y (of the pixel) and Path for merging, forming a unique identifier for each pixel
metadata_parquet_file = "metadata_tutorial.parquet"

# the annotation csv file, which has columns m/z, Lipids, Score. it can come from Metaspace download, ideally a paired LCMS dataset, or others
# m/z is the mass-to-charge ratio of the lipid in the reference
# Lipids is a ";"-separated string listing lipid names that result in that m/z peak
# Score is an arbitrary score on the reliability of the annotation
annotation_csv = "lipids_processed.csv"

# if available, a quantitative LCMS can help disambiguate lipid names that result in the same m/z peak (if one is largely more abundant than others). columns: Lipid, nmol_fraction_LCMS
# Lipid is the lipid name
# nmol_fraction_LCMS is the nanomolar fraction of the lipid in the LCMS sample
lcms_abundance_csv = "lcms_females_tutorial.csv"

# the name of this analysis. this is just a string that will be used to name the output files
analysis_name = "tutorial_pregnancy"

In [ ]:
# the relative paths in the pwd that we want to analyze - where the tissue masks computed in uMAIA are found
toanalyze = ['BrainAtlas/Control_Brains/female/20220411_MouseBrain_female_217G_349x316_Att30_25um',
       'BrainAtlas/Control_Brains/female/20220412_MouseBrain_female_217B_374x286_Att30_25um',
       'BrainAtlas/Control_Brains/female/20220416_MouseBrain_female_217D_447x332_Att30_25um',
       'BrainAtlas/Control_Brains/female/20220420_MouseBrain_female_217F_383x296_Att30_25um',
       'BrainAtlas/Control_Brains/female/20220617_MouseBrain_214_A_386x291_Att30_25um',
       'BrainAtlas/Control_Brains/female/20220627_MouseBrain_214_F_383x313_Att30_25um',
       'BrainAtlas/Control_Brains/female/20220709_MouseBrain_214_C_421x328_Att30_25um',
       'BrainAtlas/Control_Brains/female/20220710_MouseBrain_214_E_415x292_Att30_25um',
       'BrainAtlas/Control_Brains/female/20220711_MouseBrain_214_D_447x327_Att30_25um',
       'BrainAtlas/Control_Brains/female/20220712_MouseBrain_308_A_371x297_Att30_25um',
       'BrainAtlas/Control_Brains/female/20220713_MouseBrain_308_B_399x315_Att30_25um',
       'BrainAtlas/Control_Brains/female/20220725_MouseBrain_308_F_368x284_Att30_25um',
       'BrainAtlas/Control_Brains/female/20220810_MouseBrain_308_D_405x296_Att30_25um',
       'BrainAtlas/Control_Brains/female/20220811_MouseBrain_308_E_410x292_Att30_25um',
       'PREGNANT/20240708_MouseBrain_LipidAtlas_Pregnant_Brain1_A1_395x280_25um_Att30',
       'PREGNANT/20240709_MouseBrain_LipidAtlas_Pregnant_Brain1_B1_458x337_25um_Att30',
       'PREGNANT/20240712_MouseBrain_LipidAtlas_Pregnant_Brain1_C2_459x352_25um_Att30',
       'PREGNANT/20240714_MouseBrain_LipidAtlas_Pregnant_Brain1_D1_477x346_25um_Att30',
       'PREGNANT/20240715_MouseBrain_LipidAtlas_Pregnant_Brain1_E1_454x310_25um_Att30',
       'PREGNANT/20240711_MouseBrain_LipidAtlas_Pregnant_Brain1_F1_424x338_25um_Att30',
       'PREGNANT/20240710_MouseBrain_LipidAtlas_Pregnant_Brain2_A1_380x297_25um_Att30',
       'PREGNANT/20240721_MouseBrain_LipidAtlas_Pregnant_Brain2_B1_434x310_25um_Att30',
       'PREGNANT/20240716_MouseBrain_LipidAtlas_Pregnant_Brain2_C1_445x318_25um_Att30',
       'PREGNANT/20240717_MouseBrain_LipidAtlas_Pregnant_Brain2_D1_452x333_25um_Att30',
       'PREGNANT/20240720_MouseBrain_LipidAtlas_Pregnant_Brain2_E1_436x299_25um_Att30',
       'PREGNANT/20240719_MouseBrain_LipidAtlas_Pregnant_Brain2_F1_383x288_25um_Att30',
       'PREGNANT/20240722_MouseBrain_LipidAtlas_Pregnant_Brain4_A1_383x301_25um_Att30',
       'PREGNANT/20240723_MouseBrain_LipidAtlas_Pregnant_Brain4_B1_432x321_25um_Att30',
       'PREGNANT/20240724_MouseBrain_LipidAtlas_Pregnant_Brain4_C1_467x353_25um_Att30',
       'PREGNANT/20240727_MouseBrain_LipidAtlas_Pregnant_Brain4_D1_452x307_25um_Att30',
       'PREGNANT/20240726_MouseBrain_LipidAtlas_Pregnant_Brain4_E1_436x307_25um_Att30',
       'PREGNANT/20240725_MouseBrain_LipidAtlas_Pregnant_Brain4_F1_336x273_25um_Att30']

In [ ]:
# load the information needed to run the pipeline, formatted as in this tutorial

acquisitions = pd.read_csv(acquisition_paths_csv, index_col=0)
acquisitions = acquisitions.loc[acquisitions['acqpath'].isin(toanalyze),:]

## Preprocessing

In [ ]:
prep = Preprocessing()

# in case we want instead to resume from where we left
# prep.load_msi_dataset()

In [ ]:
# calculate Moran's I for all the acquisitions. this can take hours 

morans_df = prep.calculate_moran(
    path_data=uMAIA_path,
    acquisitions = acquisitions,
    log_file="iterations_log.txt",
    morans_csv="morans_by_sec.csv"
)

# or read it from file
# moran_df = pd.read_csv("morans_by_sec.csv", index_col=0)
moran_df = morans_df
moran_df

In [ ]:
# store the uMAIA-normalized data in a convenient dataframe, masking the tissue, and adding basic acquisition-level metadata. 
# note that from here, the uMAIA-logged data is brought back to its native scale by exponentiation.
# this can take > 1h
prep.store_exp_data_metadata(
    path_data=uMAIA_path,
    acquisitions = acquisitions,
    metadata_csv=metadata_csv_file
)

# create section and pixel-aware barcodes that simplify referencing to individual pixels
prep.add_barcodes_to_adata()

# add pixel-level metadata, such as the regions within the tissue, annotated with other tools
prep.add_metadata_from_parquet(metadata_parquet_file)

# filter out by some metadata, to better remove background (eg, color of a tissue annotation where no tissue is found)
prep.filter_by_metadata("allencolor", "!='#000000'")
prep.adata

In [ ]:
# prepare the lipid name annotation table based on LIPIDMAPS and user-provided uploaded annotations eg from LC-MS
matched_table = prep.annotate_molecules( 
        user_annotation_csv=annotation_csv, # this is the table with the user-provided annotations
        ppm=5
    )

In [ ]:
# prioritize the lipids based on the LC-MS abundance in case of ties
prioritized_table = prep.abundance_prioritization_lcms(
    matched_table,
    lcms_csv=lcms_abundance_csv, # this is the table with the LC-MS abundance per lipid
    annotation_col='Lipid',
    threshold=0.8
)

In [ ]:
# prioritize different adducts of the same lipid based on total signal in case of ties
final_table = prep.prioritize_adducts_by_signal(
    path_data=uMAIA_path,
    acquisitions=acquisitions,
    prioritized_table=prioritized_table,
    annotation_col='AnnotationLCMSPrioritized',
    n_sections=5 # this is the number of sections to use for the signal prioritization. small to prototype fast, number of all sections in the study to reliably assess them all
) 
final_table

In [ ]:
# as MALDI-MSI cannot compare different lipids, we can normalize each lipid to a range of 0-1, viewing each lipid as a relative level compared to the rest of the dataset
prep.min0max1_normalize_clip()

In [ ]:
# perform feature selection based on Moran's I and variance vs mean of section-wise variances
# this can take some minutes
prep.feature_selection(
        moran_df,
        modality = "combined",  # options: "moran", "combined", "manual"
        mz_vals = None,  # if provided, these m/z values override all other criteria
        moran_threshold = 0.25,
        cluster_k = 10, # number of clusters of lipids to be searched in the space of scores for feature selection
        output_csv = "feature_scores.csv",
        remove_untrustworthy = False
    )


In [ ]:
# use the LC-MS and/or LIPIDMAPS annotations to rename the features
prep.rename_features(final_table, 
                     annotation_col="AnnotationLCMSPrioritized", 
                     min_score=4.0) # a minimum score on the annotation confidence (provided by the user for each lipid)

In [ ]:
# extract a table with metadata for each lipid (class, chain length, insaturation...)
lipid_props_df = prep.lipid_properties()
lipid_props_df

In [ ]:
# extract a table of metabolic connections among pairs of measured lipids, and annotate the corresponding enzymes
reaction_df = prep.reaction_network(lipid_props_df)
reaction_df

In [ ]:
# save the dataset for later use
prep.save_msi_dataset()

## Embedding

In [ ]:
# reload the dataset with the preprocessing operations applied
prep = Preprocessing()
prep.load_msi_dataset("analysis_msi_dataset_preprocessing_ops.h5ad")

# separate the control condition from the rest. it will be used to determine the embeddings
prep_naive = Preprocessing()
prep_naive.adata = prep.adata[prep.adata.obs['Condition'] == "naive"]

# learn the NMF embeddings using a seeding procedure based on the lipid-lipid correlation matrix. this can take some minutes
emb1 = Embedding(prep_naive)
emb1.learn_seeded_nmf_embeddings()

In [ ]:
# deploy the NMF on all acquisitions, including the control and the case conditions
prep.adata =emb1.apply_nmf_embeddings(new_adata = prep.adata)

# make an embedding object with the adata that contains the NMF embeddings, and add to it the NMF right matrix (factor_to_lipid)
emb = Embedding(prep)
emb.factor_to_lipid = emb1.factor_to_lipid

# harmonize the NMF embeddings across batches, using the covariates provided. this can take minutes to > 1h
emb.harmonize_nmf_batches(covariates=['SectionID','Sample', 'Sex', 'Condition'])

# use the harmonized NMF embeddings to approximate the original dataset for clustering-related procedures that are sensitive to batch effects
emb.approximate_dataset_harmonmf()

# make a t-SNE on top of the NMF embeddings
emb.tsne(
        perplexity = 30,
        n_iter1 = 500,
        exaggeration1 = 1.2,
        n_iter2 = 100,
        exaggeration2 = 2.5,
        init_indices = (0, 1)
    )

# save the dataset for later use and export a plot of all embeddings to pdf
emb.save_msi_dataset(plot_embeddings=True) 

## Feature recovery

In [ ]:
# load the reference image and the annotation image from a matched tissue reference atlas if available
reference_image = np.load("reference_image100um.npy") 
annotation_image = np.load("annotation_image100um.npy")

# create a postprocessing object
postproc = Postprocessing(emb, moran_df, reference_image, annotation_image)

# restore lipids that are measured reliably in a fraction of sections. this can take several hours
postproc.xgboost_feature_restoration(
        moran_threshold=0.25,
        usage_threshold=0.25,
        usage_top_n=3,
        usage_sum_threshold=2,
        acquisitions=acquisitions,
        min_sections_for_training=3,
        train_section_indices=(0, 2),
        val_section_index=1,
        valid_pearson_threshold=0.4,
        output_model_dir="xgbmodels",
        metrics_csv="metrics_imputation_df.csv",
        xgb_params=None
    ) 

In [ ]:
# add to the adata a lipidome matrix that includes both the xgboosted and the raw lipids, without redundancy (1 entry per unique lipid name)
postproc.create_lipidome_matrix(final_table, min_score=4.0)

postproc.analysis_name = "pregnancy_tutorial"

# save the dataset for later use
postproc.save_msi_dataset()

# plot all the lipids with their names for all the sections to pdf, with the naming quality scores
postproc.plot_all_annotated_lipids(final_table)

## Clustering

In [ ]:
emb = Embedding(None)
emb.load_msi_dataset()

In [ ]:
# load the dataset for the naive condition to learn the EUCLID clustering on it, and later apply it to the case condition
import copy
emb_naive = copy.copy(emb)
emb_naive.adata = emb_naive.adata[emb_naive.adata.obs['Condition'] == "naive"]
clust = Clustering(emb_naive)

In [ ]:
# first do Leiden clustering as a starting point. this can take > 1h
#clust.leiden_nmf(resolution=8.0, key_added="X_Leiden")

# then do EUCLID clustering. this can take several hours. for reference, for 1.5 million pixels it takes about 24 hours. for 300k pixels, it takes about 2 hours
root_node, clusteringLOG = clust.learn_euclid_clustering(
                                K=60,
                                min_voxels=150,
                                min_diff_lipids=2,
                                min_fc=0.2,
                                pthr=0.05,
                                ACCTHR=0.6,
                                max_depth=5, 
                                min_nonzero_sections=2, 
                                gaussian_sigma=1.8,  
                                peak_count_threshold=3, 
                                peak_ratio_threshold=1.4,
                                combinations=20, do_plotting=True
                                )

In [ ]:
# add the clustering results to the adata object
clust.add_clustering_to_adata(clusteringLOG)

# and compare the two
#clust.compare_leiden_lipizone()

# assign colors to the clusters in a structured way
clust.assign_cluster_colors(clusteringLOG)

# give a name to the lipizones based on the anatomy or other available metadata
clust.name_lipizones_anatomy('acronym', 'lipizone')
clust.adata

In [ ]:
# plot distribution of # pixels per lipizone
clust.plot_distribution_of_pixels_per_lipizone()

# plot histogram of number of lipids per lipizone
clust.plot_histogram_of_lipids_per_lipizone()

## Label transfer

In [ ]:
# apply the EUCLID clustering learnt on the control condition to the whole dataset. this can take > 1 hour
clust2 = Clustering(emb)
clusteringLOG = clust2.apply_euclid_clustering(root_node, clust2.adatamaia, clust2.standardized_embeddings_GLOBAL)

clust2.add_clustering_to_adata(clusteringLOG)

# plot the clusters to pdf one by one for inspection - takes about 1 hour
clust2.clusters_to_pdf(output_folder="lipizones_output", pdf_filename="clusters_combined.pdf")

# transfer the color and name from the clustered control data onto the full label-transferred dataset
clust2.paint_lipizones(clust.adata)

# save the dataset to file for future use
clust2.save_msi_dataset()
clust2.adata.obs['lipizone_names']

## Downstream operations and visualizations

In [ ]:
# bring the clustering information into the postprocessing object since all other operations will operate on it: load and combine

clust = Clustering(None)
clust.load_msi_dataset()

postproc = Postprocessing(None, None, analysis_name="pregnancy_tutorial")
postproc.load_msi_dataset()

postproc.merge_adata_objects(clust.adata)

## Lipid Programs

## Other postprocessing operations

In [ ]:
# evaluate interindividual variability
icc_df = postproc.evaluate_interindividual_variation(subject_col="Sample")
icc_df.sort_values()[::-1]

In [ ]:
# 3D rendering. this can take tens of minutes per lipid
postproc.reference_image = np.load("reference_image100um.npy")
postproc.annotation_image = np.load("annotation_image100um.npy")

postproc.anatomical_interpolation(['HexCer 40:0;O2'], output_dir="3d_interpolated_native")

In [ ]:
# compare parcellations
colocalization_parcellations = postproc.compare_parcellations('acronym', 'lipizone_names', M=20, output_pdf="allen_vs_lipizones.pdf")

# umap of lipids
postproc.umap_molecules(color_df = lipid_props_df)

# tSNE of lipizones
postproc.tsne_lipizones()

# find spatial modules for selected sections or acronyms
postproc.spatial_modules([1, 2, 3], LA="VISpl1", LB="BMAa")

## Visualizations

In [ ]:
from euclid_msi.plotting import Plotting

plotter = Plotting(postproc.adata, analysis_name="tutorial_pregnancy")

plotter.plot_lipid_class_pie(lipid_props_df)

plotter.plot_chainlength_insaturation_hist(lipid_props_df)

In [ ]:
plotter.plot_lipid_distribution(
    lipid="PA 34:1",
    #section_filter=[1, 2],
    #metadata_filter={"division": ["Isocortex"]},
    #lipizone_filter={"lipizone_names": ["ZoneA"]},
    #x_range=(0, 100),
    #y_range=(0, 50),
    layout=None,
    show_contours=True
)

In [ ]:
plotter.plot_lipid_distribution(
    lipid="HexCer 42:2;O2",
    section_filter=[1, 2],
    #metadata_filter={"division": ["Isocortex"]},
    lipizone_filter={"lipizone_names": ["Broad_14"]},
    #x_range=(0, 100),
    #y_range=(0, 50),
    layout=None,
    show_contours=False
)

In [ ]:
plotter.plot_embeddings(key = "X_NMF",
                        currentProgram = 8)

plotter.plot_embeddings(key = "X_Harmonized",
                        currentProgram = 8)

In [ ]:
plotter.plot_embeddings(key = "X_restored", # see one xgboosted peak
                        currentProgram = 0)

plotter.plot_embeddings(key = "peaks",  # see one raw m/z peak
                        currentProgram = 10)

In [ ]:
plotter.plot_lipizones( 
    lipizone_col="lipizone_color",
    section_col="SectionID",
    #level_filter="level_1",
)

In [ ]:
plotter.plot_global_lipidomic_similarity(coloring="class")

In [ ]:
plotter.plot_tsne(
    attribute="PA 34:1",
    attribute_type="lipid",
    cmap="plasma",
    #point_size=0.5,
)

plotter.plot_tsne(
    attribute="peaks",
    attribute_type="peak",
    program_index=1,
    cmap="viridis"
)

plotter.plot_tsne(
    attribute=None,               
    attribute_type="program",
    program_key="X_Harmonized",
    program_index=2,
    cmap="PuOr"
)

plotter.plot_tsne(
    attribute=None,               
    attribute_type="program",
    program_key="X_restored",
    program_index=0,
    cmap="plasma"
)

plotter.plot_tsne(
    attribute="division",
    attribute_type="categorical",
    cmap="tab20"
)

plotter.plot_tsne(
    attribute="lipizone_color",
    attribute_type="categorical"
)

plotter.plot_tsne(
    attribute="allencolor",
    attribute_type="categorical"
)

In [ ]:
# OLO-sorted lipid by lipizone heatmap

plotter.plot_olosorted_lipid_lipizone(show_inline=True)

In [ ]:
# plot lipids in groups of 3 with proper color mixing, in a grid

lipid_names = np.random.choice(postproc.adata.var_names, size=6, replace=False)
section_ids = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
plotter.plot_lipids_rgb_grid(lipid_names, section_ids, group_size=3, output_file="brain_sections_scatter_3.png")

In [ ]:
plotter.plot_mosaic(section_id=7.0)

In [ ]:
# make a movie of all lipids traversing the tissue
plotter.create_lipids_movie(
    pc_list=["HexCer 40:0;O2"], 
    interp_dir="3d_interpolated_native",                     
    movie_dir="3d_interpolated_native",                      
    final_movie_path="lipids_movie.mp4",    
    grid_cols=4,                             
    grid_rows=3                             
)

# make a raining lipizones movie to see their crazy patterns!
plotter.make_lipizone_rain_movie(section_id=12)

In [ ]:
# make a movie to see how the hierarchical binary splitter behaved
plotter.make_splitter_movie(section_id=12)

In [ ]:
# make interactive plotly visualizers - this is still WIP!

fig = plotter.all_sections_lipid_visualizer(
    lipid_name='HexCer 42:2;O2',  
    section_key='Section', 
    x_key='z_index',       
    y_key='y_index',      
    n_cols=12             
)
fig.show()

fig = plotter.all_sections_lipizone_visualizer(
    section_key='Section',          
    x_key='z_index',               
    y_key='y_index',                
    lipizone_key='lipizone',       
    lipizone_color_key='lipizone_color',
    n_cols=12                      
)
fig.show()

fig = plotter.treemap_lipizones_adata(
    path=None,                    
    lipizone_key='lipizone',      
    color_key='lipizone_color',   
    maxdepth=2                    
)
fig.show()

lipid_array = np.load("./3d_interpolated_native/HexCer 40:0;O2_interpolation_log.npy")
fig = plotter.compute_3d_lipid_figure_from_array(
    lipid_array=lipid_array,
    annotation_array=None,                   
    set_id_regions=None,                      
    downsample_factor=2,                      
    opacity=0.4,                              
    surface_count=15,                         
    colorscale="Inferno",                    
    root_decrease_dimensionality_factor=4     
)
fig.write_html(
    "lipid_volume3d.html",
    include_plotlyjs="cdn",    
    full_html=True,            
    auto_open=False)           

In [ ]:
# 3d rendering of lipizones and their export as plotly renders - also a bit WIP
test_cluster = "AHN"  

plotter.create_marching_cubes_volumes(
    label_col='lipizone_names',
    color_col='lipizone_color',
    x_col='x_index', y_col='y_index', z_col='z_index',
    out_dir="3d_lipizone_renders",
    test_cluster=test_cluster
)


plotter.plot_marching_cubes_plotly(
    out_dir="3d_lipizone_renders",
    color_col='lipizone_color',
    opacity_mesh=0.8,
    html_file="lipizone_3d.html",
    volume_trace=reference_image,
    test_cluster=test_cluster
)

## Analyze differential lipids

In [ ]:
# eg two regions, barplot and spatial local

case_control = CaseControlAnalysis(postproc.adata, analysis_name="tutorial_pregnancy")
del postproc

results = case_control.differential_lipids(
    samples_to_keep=['Female1', 'Female2', 'Female3'],
    group_col='acronym', 
    group1='MPN',
    group2='AHN',
    lipid_props_df=lipid_props_df,
    min_fc=0.2,
    pthr=0.05,
    show_inline=True,
    output_filename='differential_results.pdf'
)

## Train a Bayesian model for case-control analyses

In [ ]:
# detect differences between case and control with a model aware of interindividual variability and batch effects
# the output is a "shift", or net difference, which can be readily converted to a log2 fold change, while testing 
# in a Bayesian fashionfor the confidence that the magnitude of change is at least above a desired threshold
# this can take several hours to run!
results = case_control.run_case_control_analysis(
     lipids_to_analyze=["HexCer 40:2;O2_1", "SM 42:2;O2", "PG 40:6"],
     num_epochs=2500,
     learning_rate=0.01,
     normalize_percentiles=(0.5, 99.5),
     x_col="x", y_col="y", sectionid_col="SectionID", sample_col="Sample", condition_col="Condition", supertype_col="supertype"
)

## Analyze the condition-driven lipidomic shifts

In [ ]:
shift, baseline, foldchange = case_control.summarize_case_control_results(
    lipids_to_analyze=["HexCer 40:2;O2_1", "SM 42:2;O2", "PG 40:6"],
    normalize_percentiles=(0.5, 99.5),
    output_prefix="pregnancy"
)

In [ ]:
upreg, downreg, expressed, shifted, mean_score, ci_lowers, ci_uppers = case_control.summarize_xsupertypes(
    lipids_to_analyze=["HexCer 40:2;O2_1", "SM 42:2;O2", "PG 40:6"],
    output_prefix="pregnancy"
)

In [ ]:
# to be tested when more lipids are available

#case_control.plot_comodulation_heatmap(
#    shift, baseline, expressed, shifted, lipid_props_df,threshold=0.4,
#    #todrop_supertypes=['11211222', '11222222'], 
#    #todrop_lipids=['TG 72:9', 'TG 67:2'],
#    k_row=2
#)

#mod_scores = case_control.compute_edge_modulation_scores(foldchanges, metabolicmodule, comodulation_clusters)
#case_control.plot_modulation_thumbnails(mod_scores, metabolicmodule, ddf)